In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf
import os
import json
pd.options.mode.chained_assignment = None #don't show warning flags
%matplotlib inline

from IPython.display import display, HTML
display(HTML(
    '<style>'
        '#notebook { padding-top:0px !important; } ' 
        '.container { width:97% !important; } '
        '.end_space { min-height:0px !important; } '
    '</style>'
))

pd.set_option('display.max_columns', None)
sns.set(rc={'figure.figsize':(16.7,4.27)})

pd.options.display.float_format = '{:.0f}'.format #surpress scientific notation

In [2]:
# Tournament data gotten from 
# https://github.com/Badaro/MTGODecklistCache/tree/master/Tournaments/mtgo.com

In [97]:
### TOURNAMENT DATA ###
def normalize_decklist(tournament_decklist, record_path):
    cleaned_decklist = pd.json_normalize(tournament_decklist['Decks'], meta=['Date', 'Player', 'Result'], record_path = record_path)
    cleaned_decklist['deck_type'] = record_path
    cleaned_decklist['join_field'] = 1
    return cleaned_decklist

def get_tournament_info(decklist_json):
    tournament_info = pd.json_normalize(decklist_json['Tournament'])
    tournament_info.drop(['Uri'],axis = 1, inplace = True)
    tournament_info['join_field'] = 1 
    return tournament_info

def cleanup_decklist_name(deck_data):
    deck_data.rename({'Count':'card_count', 'CardName':'card_name', 'Date':'tournament_date', 
                        'Player':'player_name', 'Result':'tournament_result', 'Name' : 'tournament_name'}, axis = 1, inplace = True)
    
def cleanup_decklist_order(deck_data):
    deck_data = deck_data[['card_name', 'card_count', 'deck_type', 'tournament_name', 'tournament_result', 'tournament_date', 'player_name']]
    return deck_data

def cleanup_decklist_data(deck_data):
    cleanup_decklist_name(deck_data)
    deck_data.drop(['join_field'],axis=1, inplace=True)
    deck_data = cleanup_decklist_order(deck_data)
    return deck_data

def get_decklist_data(tournament_decklist):
    main_decklist = normalize_decklist(tournament_decklist, 'Mainboard')
    side_decklist = normalize_decklist(tournament_decklist, 'Sideboard')
    overall_decklist = pd.concat([main_decklist, side_decklist])
    overall_decklist.drop(['Date'],axis=1, inplace=True) #gotten from tournament data
    
    tournament_info = get_tournament_info(tournament_decklist)
    overall_decklist = pd.merge(overall_decklist, tournament_info, on=['join_field'], how='inner')
    overall_decklist = cleanup_decklist_data(overall_decklist)
    return overall_decklist

def create_placement(placement_stop):
    keep_tournament_result = ['7-0', '6-0', '5-0','4-0', '6-1', '5-1', '4-1', '5-2', '1st Place', '2nd Place', '3rd Place'] #can't use fuzzy matching as it'll bring things like 18th place in
    placement = []
    for i in range(4, placement_stop + 1):
        keep_tournament_result.append(str(i) + 'th Place')
    return keep_tournament_result
        
def filter_tournament_result(decklist_data):
    keep_tournament_result = create_placement(8)
    decklist_data = decklist_data[decklist_data['tournament_result'].isin(keep_tournament_result)]
    return decklist_data

def get_tournament_decklists(mtg_format, directory):
    overall_decklist_data = pd.DataFrame()

    current_path = os.getcwd()
    current_path = current_path + "\\" + directory
    
    for root, dir_names, file_names in os.walk(current_path):
        print('checking results in: {}'.format(root))
        for f in file_names:
            if mtg_format in f:
                print(" * Adding results from: {}".format(f))
                fname = os.path.join(root, f)
                tournament_decklist = open(fname, encoding= 'utf-8') #some cards have accents in name
                tournament_decklist = json.load(tournament_decklist)
                tournament_decklist = get_decklist_data(tournament_decklist)
                tournament_decklist['tournament_full_name'] = f[:-5] #remove .json
                #tournament_decklist = filter_tournament_result(tournament_decklist) # may be better to not artifically limit dataset
                overall_decklist_data = pd.concat([overall_decklist_data, tournament_decklist])
        print("")
    return overall_decklist_data

def create_tournament_date():
    tournament_decklists['tournament_date'] = pd.to_datetime(tournament_decklists['tournament_date']).dt.normalize()
    tournament_decklists['tournament_date'] = tournament_decklists['tournament_date'].dt.tz_localize(None)

def remove_blank_tournament_result(tournament_decklists):
    tournament_decklists = tournament_decklists.loc[(tournament_decklists['tournament_result'].notnull())]
    tournament_decklists = tournament_decklists.loc[(tournament_decklists['tournament_result'] != '')]
    return tournament_decklists

def create_ordinal_placement(placement_start, placement_stop):
    #this only works up to 20th place, worth fixing at some point !!
    ordinal_placement = []
    for i in range(placement_start, placement_stop + 1):
        ordinal_placement.append(str(i) + 'th Place')
    return ordinal_placement

### SCRYFALL ###
def get_scryfall_data():
    #data via https://scryfall.com/docs/api/bulk-data Default Cards; bulk data used to prevent too many API calls
    print("Cleaning up JSON data")
    scryfall_json_name = scryfall_data_name + '.json'
    scryfall = open(scryfall_json_name, encoding="utf8")
    scryfall = json.load(scryfall)
    scryfall = pd.json_normalize(scryfall)
    return scryfall
    
def cleanup_scryfall(scryfall):
    scryfall = scryfall[['name', 'mana_cost', 'released_at', 'cmc', 'colors', 'rarity', 'type_line', 'oracle_text', 'power', 'toughness', 'legalities.modern', 'set_name', 'set_type', 'set']]
    scryfall.rename({'name':'card_name'}, axis = 1, inplace = True)
    return scryfall

def scryfall_date_cutoff(scryfall, cutoff_date):
    scryfall = scryfall.loc[(scryfall['released_at'] < cutoff_date)]
    return scryfall

def get_modern_legal_set(modern_legal_set_file_name):
    #taken via https://mtg.fandom.com/wiki/Modern
    modern_legal_set_file_name = modern_legal_set_file_name + '.csv'
    modern_legal_sets = pd.read_csv(modern_legal_set_file_name)
    return modern_legal_sets

def map_rarity_to_code():
    #we want to map as we'll min the rarity so we know it's current rarity
    scryfall['rarity_code'] = 0
    scryfall['rarity_code'] = scryfall['rarity'].map({'common': 1, 'uncommon': 2, 'rare': 3, 'mythic': 4})

def map_code_to_rarity():
    #We got the min values for rarity as numbers and now we want them as words again 
    scryfall['rarity'] = 'Unknown/Other'
    scryfall['rarity'] = scryfall['rarity_code'].map({1: 'common', 2: 'uncommon', 3: 'rare', 4: 'mythic'})
    
### GENERIC CLEANUP FUNCTIONS ###
def fix_modual_card_names(df):
    df['card_name_normalized'] = df.card_name.str.extract('^([^//]*).*', expand=False)
    df['card_name_normalized'] = df['card_name_normalized'].str.strip()
    
def cleanup_date(df, field):
    df[field] = pd.to_datetime(df[field]).dt.normalize()
    df[field] = df[field].dt.tz_localize(None)

def apply_int_type(int_columns, df):
    df[int_columns] = df[int_columns].apply(pd.to_numeric, errors='coerce')
    
### CARD SCORING SYSYEM ###
def make_tournament_result_rating(tournament_decklists):
    # logic treat X-0 and Top 4 placements highest then X-1 and 5-8th, this is because Top 8 is a high ranking in MTG tournaments, then 5-2 and top 16, everything else doesn't really matter 
    # probably a cleaner way to do this but this is fine for now; tournament size is it's own value so we treat all top placements equally
    result_top = ['7-0', '6-0', '5-0', '4-0', '1st Place', '2nd Place', '3rd Place', '4th Place']
    result_high = ['6-1', '5-1', '4-1'] + create_ordinal_placement(5, 8)
    result_middle = ['5-2'] + create_ordinal_placement(9, 16)
    tournament_decklists['tournament_result_rating'] = 0
    tournament_decklists.loc[tournament_decklists['tournament_result'].isin(result_top), 'tournament_result_rating'] = 100
    tournament_decklists.loc[tournament_decklists['tournament_result'].isin(result_high), 'tournament_result_rating'] = 75
    tournament_decklists.loc[tournament_decklists['tournament_result'].isin(result_middle), 'tournament_result_rating'] = 50
    return tournament_decklists

## tournament_result_rating Functions
def make_tournament_unique():
    tournament_weight = tournament_decklists[['tournament_name', 'tournament_full_name']]
    tournament_weight = tournament_weight.drop_duplicates()
    return tournament_weight

def calculate_tournament_size_percent(tournament_weight):
    tournament_count = len(tournament_weight)
    tournament_weight = tournament_weight['tournament_name'].value_counts().reset_index(name = 'count')
    tournament_weight.rename({'index':'tournament_name'}, axis = 1, inplace = True)
    tournament_weight['total_percent'] = round(tournament_weight['count'] / tournament_count, 4)
    return tournament_weight

def make_tournament_size_rating(tournament_decklists):
    tournament_size = make_tournament_unique()
    tournament_size = calculate_tournament_size_percent(tournament_size)
    tournament_size['total_percent_invert'] = 1 - tournament_size['total_percent']
    total_percent_invert_max = max(tournament_size['total_percent_invert']) #scale based on biggest (i.e. fewest played) tournament
    tournament_size['total_percent_invert_max'] = round(tournament_size['total_percent_invert'] / total_percent_invert_max, 4)
    tournament_size['tournament_size_rating'] = 100 * tournament_size['total_percent_invert_max']
    tournament_size.drop(['total_percent', 'total_percent_invert', 'total_percent_invert_max', 'count'],axis=1, inplace=True)
    tournament_decklists = pd.merge(tournament_decklists, tournament_size, on=['tournament_name'], how='left')
    return tournament_decklists
##

def make_card_count_rating(tournament_decklists):
    #currently we want to make everything relative to the number of cards, this is in place if we change it further
    copy_multiplyer = 1
    tournament_decklists['card_count_rating'] = tournament_decklists['card_count'] * copy_multiplyer
    return tournament_decklists

def make_deck_type_rating(tournament_decklists):
    #sideboard penalty is just based on gut feeling
    tournament_decklists['deck_type_rating'] = tournament_decklists['deck_type'].map({'Mainboard': 1, 'Sideboard': .75})
    return tournament_decklists

def make_rating(tournament_decklists):
    print("Making tournament_result_rating")
    tournament_decklists = make_tournament_result_rating(tournament_decklists)
    print("Making tournament_size_rating")
    tournament_decklists = make_tournament_size_rating(tournament_decklists)
    print("Making card_count_rating")
    tournament_decklists = make_card_count_rating(tournament_decklists)
    print("Making deck_type_rating")
    tournament_decklists = make_deck_type_rating(tournament_decklists)
    return tournament_decklists

def make_rating_weight():
    #weights based on gut feeling, we want tournament size to be more impactful so it's not overshadowed by things - remember that poor placement gives you 0 points for placement and you already have to be skilled to be in the larger tournaments as is
    rating_weight = {'tournament_result_rating': 1, 'tournament_size_rating': 2.5}
    for i in rating_weight:
        tournament_decklists[i] = tournament_decklists[i] * rating_weight[i]

In [4]:
generate_data = False
tournament_csv_name = 'modern_tournament_decklists_17_18.csv'

if generate_data:
    print("Cleaning up JSON data")
    tournament_decklists = get_tournament_decklists('modern', 'deck_data') #only care about modern
    print("Writing data to CSV")
    tournament_decklists.to_csv(tournament_csv_name, encoding='utf-8', index=False)
else:
    print("Data already made, skipping writing process")
    tournament_decklists = pd.read_csv(tournament_csv_name)

tournament_decklists = remove_blank_tournament_result(tournament_decklists) 
cleanup_date(tournament_decklists, 'tournament_date')
fix_modual_card_names(tournament_decklists)

tournament_decklists

Data already made, skipping writing process


,card_name,card_count,deck_type,tournament_name,tournament_result,tournament_date,player_name,tournament_full_name,card_name_normalized
0,Blood Crypt,1,Mainboard,Modern PTQ Preliminary,5-0,2017-01-01,__Ghost__,modern-ptq-preliminary-2017-01-0110326350,Blood Crypt
1,Bloodstained Mire,1,Mainboard,Modern PTQ Preliminary,5-0,2017-01-01,__Ghost__,modern-ptq-preliminary-2017-01-0110326350,Bloodstained Mire
2,Delver of Secrets,4,Mainboard,Modern PTQ Preliminary,5-0,2017-01-01,__Ghost__,modern-ptq-preliminary-2017-01-0110326350,Delver of Secrets
3,Gitaxian Probe,3,Mainboard,Modern PTQ Preliminary,5-0,2017-01-01,__Ghost__,modern-ptq-preliminary-2017-01-0110326350,Gitaxian Probe
4,Island,2,Mainboard,Modern PTQ Preliminary,5-0,2017-01-01,__Ghost__,modern-ptq-preliminary-2017-01-0110326350,Island
...,...,...,...,...,...,...,...,...,...
1327125,Anger of the Gods,2,Sideboard,Modern League,5-0,2018-12-31,mksc,modern-league-2018-12-313520,Anger of the Gods
1327126,Ceremonious Rejection,2,Sideboard,Modern League,5-0,2018-12-31,mksc,modern-league-2018-12-313520,Ceremonious Rejection
1327127,Fevered Visions,2,Sideboard,Modern League,5-0,2018-12-31,mksc,modern-league-2018-12-313520,Fevered Visions
1327128,Ravenous Trap,3,Sideboard,Modern League,5-0,2018-12-31,mksc,modern-league-2018-12-313520,Ravenous Trap


In [5]:
scryfall_data_name = 'scryfall_default_cards'
scryfall_csv_name = scryfall_data_name + '.csv'

generate_data = False
if generate_data:
    scryfall = get_scryfall_data()
    print("Writing data to CSV")
    scryfall.to_csv(scryfall_csv_name, encoding='utf-8', index=False)
else:
    print("Data already made, skipping writing process")
    scryfall = pd.read_csv(scryfall_csv_name)

scryfall = cleanup_scryfall(scryfall) #something weird about scoping why I can't change in place
scryfall = scryfall_date_cutoff(scryfall, '2019-01-01')

scry_int_columns = ['power','toughness', 'cmc']
apply_int_type(scry_int_columns, scryfall)

cleanup_date(scryfall, 'released_at')
fix_modual_card_names(scryfall)
map_rarity_to_code()
scryfall.drop(['rarity', 'set_name', 'set_type', 'set', 'legalities.modern', 'card_name'],axis=1, inplace=True)
scryfall = scryfall.groupby(['card_name_normalized', 'mana_cost', 'cmc', 'colors', 'type_line', 'oracle_text', 'power', 'toughness'], dropna = False).agg({'rarity_code': 'min', 'released_at': 'min'}, dropna = False).reset_index()
map_code_to_rarity()
scryfall

Data already made, skipping writing process


C:\Users\david\AppData\Local\Temp\ipykernel_9708\1960687761.py:11: DtypeWarning: Columns (111) have mixed types. Specify dtype option on import or set low_memory=False.
  scryfall = pd.read_csv(scryfall_csv_name)


,card_name_normalized,mana_cost,cmc,colors,type_line,oracle_text,power,toughness,rarity_code,released_at,rarity
0,"""Ach! Hans, Run!""",{2}{R}{R}{G}{G},6.0,"['G', 'R']",Enchantment,"At the beginning of your upkeep, you may say ""...",NaN,NaN,3.0,2004-11-19,rare
1,"""Rumors of My Death . . .""",{2}{B},3.0,['B'],Enchantment,"{3}{B}, Exile a permanent you control with a L...",NaN,NaN,2.0,2017-12-08,uncommon
2,1996 World Champion,{W}{U}{B}{R}{G},5.0,"['B', 'G', 'R', 'U', 'W']",Summon Legend,Cannot be the target of spells or effects. Wor...,NaN,NaN,3.0,1996-08-14,rare
3,1997 World Championships Ad,NaN,0.0,[],Card,NaN,NaN,NaN,1.0,1997-08-13,common
4,1998 World Championships Ad,NaN,0.0,[],Card,NaN,NaN,NaN,1.0,1998-08-12,common
...,...,...,...,...,...,...,...,...,...,...,...
19372,Zurgo Bellstriker,{R},1.0,['R'],Legendary Creature — Orc Warrior,Zurgo Bellstriker can't block creatures with p...,2.0,2.0,3.0,2015-03-27,rare
19373,Zurgo Helmsmasher,{2}{R}{W}{B},5.0,"['B', 'R', 'W']",Legendary Creature — Orc Warrior,Haste\nZurgo Helmsmasher attacks each combat i...,7.0,2.0,4.0,2014-09-05,mythic
19374,Zzzyxas's Abyss,{1}{B}{B},3.0,['B'],Enchantment,"At the beginning of your upkeep, destroy all n...",NaN,NaN,3.0,2004-11-19,rare
19375,_____,{1}{U},2.0,['U'],Creature — Shapeshifter,{1}: This card's name becomes the card name of...,1.0,1.0,2.0,2004-11-19,uncommon


In [6]:
#we bring scyrfall info in now as some elements like the time the card is legal in the format could be used at the tournament level
tournament_decklists = pd.merge(tournament_decklists, scryfall, on=['card_name_normalized'], how='left')
tournament_decklists.drop(['card_name_normalized'],axis=1, inplace=True)

tournament_decklists.loc[(tournament_decklists['released_at'].isna())]

,card_name,card_count,deck_type,tournament_name,tournament_result,tournament_date,player_name,tournament_full_name,mana_cost,cmc,colors,type_line,oracle_text,power,toughness,rarity_code,released_at,rarity


In [7]:
### Weighing System for Cards ###

In [10]:
1=2
tournament_decklists_backup = tournament_decklists.copy()

In [99]:
tournament_decklists = tournament_decklists_backup.copy()
tournament_decklists = make_rating(tournament_decklists)
make_rating_weight()
tournament_decklists

Making tournament_result_rating
Making tournament_size_rating
Making card_count_rating
Making deck_type_rating


,card_name,card_count,deck_type,tournament_name,tournament_result,tournament_date,player_name,tournament_full_name,mana_cost,cmc,colors,type_line,oracle_text,power,toughness,rarity_code,released_at,rarity,tournament_result_rating,tournament_size_rating,card_count_rating,deck_type_rating
0,Blood Crypt,1,Mainboard,Modern PTQ Preliminary,5-0,2017-01-01,__Ghost__,modern-ptq-preliminary-2017-01-0110326350,NaN,0,[],Land — Swamp Mountain,({T}: Add {B} or {R}.)\nAs Blood Crypt enters ...,NaN,NaN,3,2006-05-05,rare,100,98,1,1
1,Bloodstained Mire,1,Mainboard,Modern PTQ Preliminary,5-0,2017-01-01,__Ghost__,modern-ptq-preliminary-2017-01-0110326350,NaN,0,[],Land,"{T}, Pay 1 life, Sacrifice Bloodstained Mire: ...",NaN,NaN,3,2002-10-07,rare,100,98,1,1
2,Delver of Secrets,4,Mainboard,Modern PTQ Preliminary,5-0,2017-01-01,__Ghost__,modern-ptq-preliminary-2017-01-0110326350,NaN,1,NaN,Creature — Human Wizard // Creature — Human In...,NaN,NaN,NaN,1,2011-09-30,common,100,98,4,1
3,Gitaxian Probe,3,Mainboard,Modern PTQ Preliminary,5-0,2017-01-01,__Ghost__,modern-ptq-preliminary-2017-01-0110326350,{U/P},1,['U'],Sorcery,({U/P} can be paid with either {U} or 2 life.)...,NaN,NaN,1,2011-05-13,common,100,98,3,1
4,Island,2,Mainboard,Modern PTQ Preliminary,5-0,2017-01-01,__Ghost__,modern-ptq-preliminary-2017-01-0110326350,NaN,0,[],Basic Land — Island,({T}: Add {U}.),NaN,NaN,1,1993-08-05,common,100,98,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1326239,Anger of the Gods,2,Sideboard,Modern League,5-0,2018-12-31,mksc,modern-league-2018-12-313520,{1}{R}{R},3,['R'],Sorcery,Anger of the Gods deals 3 damage to each creat...,NaN,NaN,3,2013-09-27,rare,100,15,2,1
1326240,Ceremonious Rejection,2,Sideboard,Modern League,5-0,2018-12-31,mksc,modern-league-2018-12-313520,{U},1,['U'],Instant,Counter target colorless spell.,NaN,NaN,2,2016-09-30,uncommon,100,15,2,1
1326241,Fevered Visions,2,Sideboard,Modern League,5-0,2018-12-31,mksc,modern-league-2018-12-313520,{1}{U}{R},3,"['R', 'U']",Enchantment,"At the beginning of each player's end step, th...",NaN,NaN,3,2016-04-08,rare,100,15,2,1
1326242,Ravenous Trap,3,Sideboard,Modern League,5-0,2018-12-31,mksc,modern-league-2018-12-313520,{2}{B}{B},4,['B'],Instant — Trap,If an opponent had three or more cards put int...,NaN,NaN,2,2009-10-02,uncommon,100,15,3,1


,card_name,card_count,deck_type,tournament_name,tournament_result,tournament_date,player_name,tournament_full_name,mana_cost,cmc,colors,type_line,oracle_text,power,toughness,rarity_code,released_at,rarity,tournament_result_rating,tournament_size_rating,card_count_rating,deck_type_rating
0,Blood Crypt,1,Mainboard,Modern PTQ Preliminary,5-0,2017-01-01,__Ghost__,modern-ptq-preliminary-2017-01-0110326350,NaN,0,[],Land — Swamp Mountain,({T}: Add {B} or {R}.)\nAs Blood Crypt enters ...,NaN,NaN,3,2006-05-05,rare,100,245,1,1
1,Bloodstained Mire,1,Mainboard,Modern PTQ Preliminary,5-0,2017-01-01,__Ghost__,modern-ptq-preliminary-2017-01-0110326350,NaN,0,[],Land,"{T}, Pay 1 life, Sacrifice Bloodstained Mire: ...",NaN,NaN,3,2002-10-07,rare,100,245,1,1
2,Delver of Secrets,4,Mainboard,Modern PTQ Preliminary,5-0,2017-01-01,__Ghost__,modern-ptq-preliminary-2017-01-0110326350,NaN,1,NaN,Creature — Human Wizard // Creature — Human In...,NaN,NaN,NaN,1,2011-09-30,common,100,245,4,1
3,Gitaxian Probe,3,Mainboard,Modern PTQ Preliminary,5-0,2017-01-01,__Ghost__,modern-ptq-preliminary-2017-01-0110326350,{U/P},1,['U'],Sorcery,({U/P} can be paid with either {U} or 2 life.)...,NaN,NaN,1,2011-05-13,common,100,245,3,1
4,Island,2,Mainboard,Modern PTQ Preliminary,5-0,2017-01-01,__Ghost__,modern-ptq-preliminary-2017-01-0110326350,NaN,0,[],Basic Land — Island,({T}: Add {U}.),NaN,NaN,1,1993-08-05,common,100,245,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1326239,Anger of the Gods,2,Sideboard,Modern League,5-0,2018-12-31,mksc,modern-league-2018-12-313520,{1}{R}{R},3,['R'],Sorcery,Anger of the Gods deals 3 damage to each creat...,NaN,NaN,3,2013-09-27,rare,100,37,2,1
1326240,Ceremonious Rejection,2,Sideboard,Modern League,5-0,2018-12-31,mksc,modern-league-2018-12-313520,{U},1,['U'],Instant,Counter target colorless spell.,NaN,NaN,2,2016-09-30,uncommon,100,37,2,1
1326241,Fevered Visions,2,Sideboard,Modern League,5-0,2018-12-31,mksc,modern-league-2018-12-313520,{1}{U}{R},3,"['R', 'U']",Enchantment,"At the beginning of each player's end step, th...",NaN,NaN,3,2016-04-08,rare,100,37,2,1
1326242,Ravenous Trap,3,Sideboard,Modern League,5-0,2018-12-31,mksc,modern-league-2018-12-313520,{2}{B}{B},4,['B'],Instant — Trap,If an opponent had three or more cards put int...,NaN,NaN,2,2009-10-02,uncommon,100,37,3,1


In [101]:
tournament_decklists['rating_score'] = (tournament_decklists['tournament_result_rating'] + tournament_decklists['tournament_size_rating']) * tournament_decklists['card_count_rating'] * tournament_decklists['deck_type_rating']

In [117]:
card_score = tournament_decklists.groupby(['card_name', 'mana_cost', 'cmc', 'colors', 'type_line', 'oracle_text', 'power', 'toughness', 'rarity_code', 'rarity', 'released_at' ], dropna = False).agg({'card_count': 'sum', 'rating_score': 'sum'}, dropna = False).reset_index()
card_score

,card_name,mana_cost,cmc,colors,type_line,oracle_text,power,toughness,rarity_code,rarity,released_at,card_count,rating_score
0,Abandoned Sarcophagus,{3},3,[],Artifact,You may cast spells that have a cycling abilit...,NaN,NaN,3,rare,2017-07-14,2,206
1,Abbot of Keral Keep,{1}{R},2,['R'],Creature — Human Monk,Prowess (Whenever you cast a noncreature spell...,2,1,3,rare,2015-07-17,26,4275
2,Abrade,{1}{R},2,['R'],Instant,Choose one —\n• Abrade deals 3 damage to targe...,NaN,NaN,2,uncommon,2017-07-14,8231,1074816
3,Abrupt Decay,{B}{G},2,"['B', 'G']",Instant,This spell can't be countered.\nDestroy target...,NaN,NaN,3,rare,2012-10-05,6823,1141499
4,Abundant Growth,{G},1,['G'],Enchantment — Aura,Enchant land\nWhen Abundant Growth enters the ...,NaN,NaN,1,common,2012-05-04,91,14172
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2976,Zombie Infestation,{1}{B},2,['B'],Enchantment,Discard two cards: Create a 2/2 black Zombie c...,NaN,NaN,2,uncommon,2001-10-01,23,5673
2977,Zulaport Cutthroat,{1}{B},2,['B'],Creature — Human Rogue Ally,Whenever Zulaport Cutthroat or another creatur...,1,1,2,uncommon,2015-10-02,40,7337
2978,Zur the Enchanter,{1}{W}{U}{B},4,"['B', 'U', 'W']",Legendary Creature — Human Wizard,"Flying\nWhenever Zur the Enchanter attacks, yo...",1,4,3,rare,2006-07-21,3,899
2979,Zur's Weirding,{3}{U},4,['U'],Enchantment,Players play with their hands revealed.\nIf a ...,NaN,NaN,3,rare,1995-06-03,1,103


In [118]:
card_score['rating_score_rank'] = card_score['rating_score'].rank(ascending = False)

In [119]:
card_score = card_score.sort_values('rating_score_rank')


In [128]:
card_score.loc[(card_score['type_line'] == 'Land')].head(20)

,card_name,mana_cost,cmc,colors,type_line,oracle_text,power,toughness,rarity_code,rarity,released_at,card_count,rating_score,rating_score_rank
252,Bloodstained Mire,NaN,0,[],Land,"{T}, Pay 1 life, Sacrifice Bloodstained Mire: ...",NaN,NaN,3,rare,2002-10-07,41877,7304122,5
2941,Wooded Foothills,NaN,0,[],Land,"{T}, Pay 1 life, Sacrifice Wooded Foothills: S...",NaN,NaN,3,rare,2002-10-07,35206,6136622,7
2177,Scalding Tarn,NaN,0,[],Land,"{T}, Pay 1 life, Sacrifice Scalding Tarn: Sear...",NaN,NaN,3,rare,2009-10-02,24595,4443849,12
899,Flooded Strand,NaN,0,[],Land,"{T}, Pay 1 life, Sacrifice Flooded Strand: Sea...",NaN,NaN,3,rare,2002-10-07,23345,4162548,16
2927,Windswept Heath,NaN,0,[],Land,"{T}, Pay 1 life, Sacrifice Windswept Heath: Se...",NaN,NaN,3,rare,2002-10-07,23441,4125894,18
1209,Horizon Canopy,NaN,0,[],Land,"{T}, Pay 1 life: Add {G} or {W}.\n{1}, {T}, Sa...",NaN,NaN,3,rare,2007-05-04,20581,3520101,22
218,Blackcleave Cliffs,NaN,0,[],Land,Blackcleave Cliffs enters the battlefield tapp...,NaN,NaN,3,rare,2010-10-01,20268,3418271,23
2805,Verdant Catacombs,NaN,0,[],Land,"{T}, Pay 1 life, Sacrifice Verdant Catacombs: ...",NaN,NaN,3,rare,2009-10-02,18781,3372968,24
981,Ghost Quarter,NaN,0,[],Land,"{T}: Add {C}.\n{T}, Sacrifice Ghost Quarter: D...",NaN,NaN,2,uncommon,2006-05-05,17137,3066756,31
1908,Polluted Delta,NaN,0,[],Land,"{T}, Pay 1 life, Sacrifice Polluted Delta: Sea...",NaN,NaN,3,rare,2002-10-07,16441,3016401,32


In [147]:
card_score.loc[(card_score['oracle_text'].str.contains('you may pay 2 life.', na=False))].head(30)

,card_name,mana_cost,cmc,colors,type_line,oracle_text,power,toughness,rarity_code,rarity,released_at,card_count,rating_score,rating_score_rank
2456,Stomping Ground,NaN,0,[],Land — Mountain Forest,({T}: Add {R} or {G}.)\nAs Stomping Ground ent...,NaN,NaN,3,rare,2006-02-03,19181,3368860,25
2438,Steam Vents,NaN,0,[],Land — Island Mountain,({T}: Add {U} or {R}.)\nAs Steam Vents enters ...,NaN,NaN,3,rare,2006-02-03,18305,3208715,29
242,Blood Crypt,NaN,0,[],Land — Swamp Mountain,({T}: Add {B} or {R}.)\nAs Blood Crypt enters ...,NaN,NaN,3,rare,2006-05-05,17361,2978768,33
2134,Sacred Foundry,NaN,0,[],Land — Mountain Plains,({T}: Add {R} or {W}.)\nAs Sacred Foundry ente...,NaN,NaN,3,rare,2005-10-07,10435,1831371,84
1134,Hallowed Fountain,NaN,0,[],Land — Plains Island,({T}: Add {W} or {U}.)\nAs Hallowed Fountain e...,NaN,NaN,3,rare,2006-05-05,10064,1788124,93
2578,Temple Garden,NaN,0,[],Land — Forest Plains,({T}: Add {G} or {W}.)\nAs Temple Garden enter...,NaN,NaN,3,rare,2005-10-07,9055,1596171,107
1822,Overgrown Tomb,NaN,0,[],Land — Swamp Forest,({T}: Add {B} or {G}.)\nAs Overgrown Tomb ente...,NaN,NaN,3,rare,2005-10-07,7969,1431678,121
2888,Watery Grave,NaN,0,[],Land — Island Swamp,({T}: Add {U} or {B}.)\nAs Watery Grave enters...,NaN,NaN,3,rare,2005-10-07,6225,1155488,140
300,Breeding Pool,NaN,0,[],Land — Forest Island,({T}: Add {G} or {U}.)\nAs Breeding Pool enter...,NaN,NaN,3,rare,2006-05-05,4427,768828,204
1061,Godless Shrine,NaN,0,[],Land — Plains Swamp,({T}: Add {W} or {B}.)\nAs Godless Shrine ente...,NaN,NaN,3,rare,2006-02-03,3140,610732,242


In [140]:
card_score.loc[(card_score['card_name'] == 'Steam Vents')].head(20)

,card_name,mana_cost,cmc,colors,type_line,oracle_text,power,toughness,rarity_code,rarity,released_at,card_count,rating_score,rating_score_rank
2438,Steam Vents,NaN,0,[],Land — Island Mountain,({T}: Add {U} or {R}.)\nAs Steam Vents enters ...,NaN,NaN,3,rare,2006-02-03,18305,3208715,29


In [125]:
card_score.head(20)

,card_name,mana_cost,cmc,colors,type_line,oracle_text,power,toughness,rarity_code,rarity,released_at,card_count,rating_score,rating_score_rank
1470,Lightning Bolt,{R},1,['R'],Instant,Lightning Bolt deals 3 damage to any target.,NaN,NaN,1,common,1993-08-05,63524,10611355,1
1297,Island,NaN,0,[],Basic Land — Island,({T}: Add {U}.),NaN,NaN,1,common,1993-08-05,49234,8683267,2
1654,Mountain,NaN,0,[],Basic Land — Mountain,({T}: Add {R}.),NaN,NaN,1,common,1993-08-05,48817,8387822,3
907,Forest,NaN,0,[],Basic Land — Forest,({T}: Add {G}.),NaN,NaN,1,common,1993-08-05,45738,7795938,4
252,Bloodstained Mire,NaN,0,[],Land,"{T}, Pay 1 life, Sacrifice Bloodstained Mire: ...",NaN,NaN,3,rare,2002-10-07,41877,7304122,5
1851,Path to Exile,{W},1,['W'],Instant,Exile target creature. Its controller may sear...,NaN,NaN,2,uncommon,2009-02-06,41379,6941992,6
2941,Wooded Foothills,NaN,0,[],Land,"{T}, Pay 1 life, Sacrifice Wooded Foothills: S...",NaN,NaN,3,rare,2002-10-07,35206,6136622,7
2234,Serum Visions,{U},1,['U'],Sorcery,Draw a card. Scry 2.,NaN,NaN,1,common,2004-01-01,32277,5813387,8
2636,Thoughtseize,{B},1,['B'],Sorcery,Target player reveals their hand. You choose a...,NaN,NaN,3,rare,2007-10-12,34102,5651602,9
835,Fatal Push,{B},1,['B'],Instant,Destroy target creature if it has mana value 2...,NaN,NaN,2,uncommon,2017-01-01,27627,4714287,10


In [ ]:
# type of tournament
# number of copies
# main vs side deck
# was the card eventually banned
# length of time card was legal in the format (so new cards)
# card type (empthasis on lands)
# mana value
# color
# number of decks the card is in
# card clustering?
# if a card was banned and unbanned and banned again need to have on list multiple times with legal date and legal end date

In [ ]:
### Code Holding ###
1=2
json_file_name = 'scryfall_default_cards.json'
#data via https://scryfall.com/docs/api/bulk-data; bulk data used to prevent too many API calls
test_scry = open(json_file_name, encoding="utf8")
test_scry = json.load(test_scry)
test_scry = pd.json_normalize(test_scry)

test_scry['lang'].unique()


In [ ]:
#if we want to factor in ban into weight of model (not doing right now to keep V1 simple)

### MODERN BAN LIST ###
def modern_ban_cutoff(modern_ban, cutoff_date):
    modern_ban = modern_ban.loc[(modern_ban['ban_date'] < cutoff_date)]
    modern_ban['unban_date'] = modern_ban['unban_date'].where(modern_ban['unban_date'] < cutoff_date, other = pd.NaT)
    return modern_ban

modern_ban_file_name = 'modern_ban_history_june_2024.csv'
#Golgari Grave-Troll appears twice as it was banned and unbanned; as such it's treated as two instances of the card
modern_ban = pd.read_csv(modern_ban_file_name)

cleanup_date(modern_ban, 'ban_date')
cleanup_date(modern_ban, 'unban_date')

modern_ban = modern_ban_cutoff(modern_ban, '2019-01-01')
modern_ban

modern_ban.head()

scryfall = pd.merge(scryfall, modern_ban, on=['card_name'], how='left')
scryfall.head()

scryfall.loc[(scryfall['card_name'] == 'Lightning Bolt')]

scryfall.loc[(scryfall['card_name'] == 'Bloodbraid Elf')]

In [ ]:
#if I add time legal in modern so we only include products that change modern legality
modern_legal_sets = get_modern_legal_set('modern_legal_set_list_june_2024')
scryfall = pd.merge(scryfall, modern_legal_sets, on=['set_name'], how='inner') #only want modern legal cards so we can reduce size by dropping not legal cards